# 웨이퍼 불량 분류 모델 학습

이 노트북은 Google Colab에서 실행하는 용도입니다. WM811K 데이터셋으로 CNN 모델을 실제로 학습한 뒤, C# UI 프로젝트에서 사용할 수 있도록 학습된 모델을 ONNX 파일로 변환합니다.


In [17]:
!pip install -q kaggle torch torchvision opencv-python pandas numpy matplotlib onnx onnxscript

## 1. Google Drive 연결

In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Kaggle API 설정

추천 방법: Colab 왼쪽 파일 패널에서 `kaggle.json` 파일을 `/content/kaggle.json` 위치에 업로드하세요.

다른 방법: 아래 셀에서 `KAGGLE_USERNAME`, `KAGGLE_KEY`를 임시로 설정할 수도 있습니다. 실제 키는 GitHub에 올리면 안 됩니다.

In [19]:
# 선택 사항: kaggle.json을 업로드하지 않을 때만 Colab 실행 중에 잠깐 입력하세요.
import os
os.environ['KAGGLE_USERNAME'] = "kimsunyeong"
os.environ['KAGGLE_KEY'] = "KGAT_d5d66278fa75651102a885c44cc020a2"

## 3. 경로 설정

Google Drive의 프로젝트 폴더명이 다르면 `PROJECT_DIR` 값을 수정하세요. 아래 셀은 흔한 두 가지 폴더 구조를 자동으로 확인합니다.

In [20]:
import os

candidate_project_dirs = [
    '/content/drive/MyDrive/Wafer-AOI-Project',
    '/content/drive/MyDrive/Wafer-AOI-Project/Wafer-AOI-Project',
]

PROJECT_DIR = candidate_project_dirs[0]
for candidate in candidate_project_dirs:
    if os.path.exists(f'{candidate}/training/export_onnx.py'):
        PROJECT_DIR = candidate
        break

SCRIPT_PATH = f'{PROJECT_DIR}/training/export_onnx.py'
print('프로젝트 경로:', PROJECT_DIR)
print('학습 스크립트 경로:', SCRIPT_PATH)

프로젝트 경로: /content/drive/MyDrive/Wafer-AOI-Project
학습 스크립트 경로: /content/drive/MyDrive/Wafer-AOI-Project/training/export_onnx.py


## 4. 모델 학습 및 ONNX 변환

아래 셀은 `training/export_onnx.py` 파일을 실행합니다. 만약 이 파일이 Google Drive에 없으면 결과 파일이 생성되지 않습니다.

처음 테스트할 때는 기본값 그대로 실행하면 됩니다. 정확도를 더 높이고 싶으면 나중에 `--epochs`와 `--max-per-class` 값을 늘리세요.

In [21]:
import os

if not os.path.exists(SCRIPT_PATH):
    raise FileNotFoundError(
        f'학습 스크립트를 찾지 못했습니다: {SCRIPT_PATH}\n'
        '해결 방법: 이 GitHub 프로젝트의 training/export_onnx.py 파일을 '
        'Google Drive의 Wafer-AOI-Project/training/ 폴더에 넣은 뒤 다시 실행하세요.'
    )

print('학습 스크립트 확인 완료:', SCRIPT_PATH)

학습 스크립트 확인 완료: /content/drive/MyDrive/Wafer-AOI-Project/training/export_onnx.py


In [22]:
!python "{SCRIPT_PATH}" --project-dir "{PROJECT_DIR}" --epochs 8 --batch-size 128 --max-per-class 1500

실행 중: kaggle datasets download -d qingyi/wm811k-wafer-map -p /content/drive/MyDrive/Wafer-AOI-Project/AI_Model/dataset
Dataset URL: https://www.kaggle.com/datasets/qingyi/wm811k-wafer-map
License(s): CC0-1.0
100% 149M/149M [00:02<00:00, 62.1MB/s]

학습에 사용할 클래스별 샘플 수:
failureType
Center       1500
Edge-Ring    1500
none         1500
Loc          1500
Edge-Loc     1500
Scratch      1193
Random        866
Donut         555
Near-full     149
Name: count, dtype: int64
사용 장치: cpu
Epoch 01/8 | 손실=1.3643 | 검증정확도=0.6611
Epoch 02/8 | 손실=0.8823 | 검증정확도=0.6787
Epoch 03/8 | 손실=0.7331 | 검증정확도=0.7235
Epoch 04/8 | 손실=0.6503 | 검증정확도=0.7743
Epoch 05/8 | 손실=0.5829 | 검증정확도=0.7899
Epoch 06/8 | 손실=0.5512 | 검증정확도=0.7830
Epoch 07/8 | 손실=0.5325 | 검증정확도=0.7367
Epoch 08/8 | 손실=0.4911 | 검증정확도=0.8035
최고 검증 정확도: 0.8035
/content/drive/MyDrive/Wafer-AOI-Project/training/export_onnx.py:291: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints viola

## 생성되는 파일

학습이 끝나면 Google Drive에 아래 파일들이 생성됩니다.

- `AI_Model/wafer_defect_model.pth`
- `AI_Model/wafer_defect_model.onnx`
- `AI_Model/wafer_labels.json`
- `Assets/TestImages/normal_wafer.png`
- `Assets/TestImages/scratch_wafer.png`

In [23]:
import os

expected_files = [
    f'{PROJECT_DIR}/AI_Model/wafer_defect_model.pth',
    f'{PROJECT_DIR}/AI_Model/wafer_defect_model.onnx',
    f'{PROJECT_DIR}/AI_Model/wafer_labels.json',
    f'{PROJECT_DIR}/Assets/TestImages/normal_wafer.png',
    f'{PROJECT_DIR}/Assets/TestImages/scratch_wafer.png',
]

for path in expected_files:
    print(('생성됨: ' if os.path.exists(path) else '없음:   ') + path)

생성됨: /content/drive/MyDrive/Wafer-AOI-Project/AI_Model/wafer_defect_model.pth
생성됨: /content/drive/MyDrive/Wafer-AOI-Project/AI_Model/wafer_defect_model.onnx
생성됨: /content/drive/MyDrive/Wafer-AOI-Project/AI_Model/wafer_labels.json
생성됨: /content/drive/MyDrive/Wafer-AOI-Project/Assets/TestImages/normal_wafer.png
생성됨: /content/drive/MyDrive/Wafer-AOI-Project/Assets/TestImages/scratch_wafer.png
